In [2]:
from pathlib import Path
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

In [3]:
from src.myio import reader,  read_gpkg, read_csv, write_parquet, read_dem, read_parquet


In [25]:
dem_root_ = reader("data_clipped")

dem_root = Path(dem_root_.full_path)

In [26]:
dem_paths = list(dem_root.rglob("*_egg2015.tif"))  # або просто "*.tif" якщо потрібно всі

[PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped/alos/alos_dem_egg2015.tif'), PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped/aster/aster_dem_egg2015.tif'), PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped/copernicus/copernicus_dеm_egg2015.tif'), PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped/fabdem/fab_dem_egg2015.tif'), PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped/nasadem/nasa_dem_egg2015.tif'), PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped/srtm/srtm_dem_egg2015.tif'), PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_clipped/tandem/tan_dem_egg2015.tif')]


In [29]:

output_dir = Path("DATA_UTM")
output_dir.mkdir(exist_ok=True)

In [30]:

# Цільова проєкція
target_crs = "EPSG:32635"  # UTM zone 35N, WGS84

for dem_path in dem_paths:
    name = dem_path.stem.replace("_egg2015", "")
    with rasterio.open(dem_path) as src:
        print(f"🔄 Обробка {name}...")

        # Обчислюємо трансформацію
        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds)

        kwargs = src.meta.copy()
        kwargs.update({
            "crs": target_crs,
            "transform": transform,
            "width": width,
            "height": height
        })

        # Шлях до нового DEM
        out_path = output_dir / f"{name}_utm32635.tif"

        # Запис нового файлу
        with rasterio.open(out_path, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=target_crs,
                    resampling=Resampling.bilinear
                )

        print(f"✅ Збережено: {out_path.name}")


🔄 Обробка alos_dem...
✅ Збережено: alos_dem_utm32635.tif
🔄 Обробка aster_dem...
✅ Збережено: aster_dem_utm32635.tif
🔄 Обробка copernicus_dеm...
✅ Збережено: copernicus_dеm_utm32635.tif
🔄 Обробка fab_dem...
✅ Збережено: fab_dem_utm32635.tif
🔄 Обробка nasa_dem...
✅ Збережено: nasa_dem_utm32635.tif
🔄 Обробка srtm_dem...
✅ Збережено: srtm_dem_utm32635.tif
🔄 Обробка tan_dem...
✅ Збережено: tan_dem_utm32635.tif
